# Ollama + LangChain: query SDR with a tool

Following
[The Dino, the Llama, and the Whale](https://deno.com/blog/the-dino-llama-and-whale),
use **ChatOllama**, **Zod**, and a **Deno notebook**. Here the schema describes
a callable Gremlin tool instead of the final answer.

Model: **granite4:7b** · Ollama: **http://localhost:11434**

Install that model on your Ollama server, authenticate with
`aws sso login --profile dsoadev`, and select the Deno kernel. The kernel must
reach both Ollama and SDR. Query results are sent to the configured Ollama
server.

In [5]:
import { ChatOllama } from "@langchain/ollama";
import { tool } from "@langchain/core/tools";
import { ChatPromptTemplate } from "@langchain/core/prompts";
import { ToolMessage } from "@langchain/core/messages";
import type { BaseMessage } from "@langchain/core/messages";
import { z } from "zod";
import { connectNotebook, ui } from "@sdr-notebook/mod";

const model = new ChatOllama({
  baseUrl: "http://localhost:11434",
  model: "granite4:7b-a1b-h",
  temperature: 0,
  maxRetries: 0,
});
const session = await connectNotebook({ profile: "dsoadev" });


## 1. Define a Zod tool schema

This starter demo offers three read queries. Zod validates the selected query
before the tool calls `session.gremlin()`, which signs and executes the SDR REST
request. Extend the catalogue to support more questions.

In [6]:
const queries = [
  'g.V().hasLabel("Study").count()',
  'g.V().hasLabel("Study").limit(5).values("name")',
  'g.V().hasLabel("Study").limit(5).project("study", "versions").by("name").by(out("has_version").count())',
] as const;

const executeGremlin = tool(
  async ({ query }) => JSON.stringify(await session.gremlin(query)),
  {
    name: "execute_gremlin",
    description:
      "Query live SDR: count all studies, list up to five study aliases, or get version counts for up to five studies. Study aliases use name; has_version links studies to versions. Samples are not exhaustive.",
    schema: z.object({
      query: z.enum(queries).describe(
        "An exact Gremlin query from the catalogue",
      ),
    }),
  },
);

const modelWithTools = model.bindTools([executeGremlin]);
ui.table(queries.map((query) => ({ query })));

query
"g.V().hasLabel(""Study"").count()"
"g.V().hasLabel(""Study"").limit(5).values(""name"")"
"g.V().hasLabel(""Study"").limit(5).project(""study"", ""versions"").by(""name"").by(out(""has_version"").count())"


## 2. Compose the prompt and model

Like the article's chain, this uses a prompt template composed with the model.
The response contains tool calls; binding a tool does not execute it. Rerun this
cell to begin a fresh question.

In [7]:
const prompt = ChatPromptTemplate.fromMessages([
  [
    "system",
    "You explore SDR using execute_gremlin. Call the tool for live data and use only its exact supported queries. Never invent results. Distinguish samples from totals. Once the requested data is available, answer concisely.",
  ],
  ["human", "{question}"],
]);
const chain = prompt.pipe(modelWithTools);
const input = {
  question:
    "Use execute_gremlin to count the studies in SDR and list up to five study aliases.",
};

const firstResponse = await chain.invoke(input, {
  signal: AbortSignal.timeout(600_000),
});
const messages: BaseMessage[] = [
  ...await prompt.formatMessages(input),
  firstResponse,
];
ui.json(firstResponse.tool_calls ?? []);

[
  {
    "name": "execute_gremlin",
    "args": {
      "query": "g.V().hasLabel(\"Study\").count()"
    },
    "id": "0d7dab98-f485-4947-b12c-e64ef24b66ec",
    "type": "tool_call"
  }
]

## 3. Execute tool calls and return their results

Preserve each assistant message and append a `ToolMessage` with the matching
call ID. Send the conversation back to the bound model until it answers. The
loop supports multiple tool calls and caps execution at four calls.

Run this cell once per invocation of the previous cell. Errors stop execution
rather than producing a fabricated answer.

In [8]:
let response = firstResponse;
const trace: Array<{ query: string; result: unknown }> = [];
const maxToolCalls = 4;

while (true) {
  if (response.invalid_tool_calls?.length) {
    throw new Error("Ollama returned malformed tool calls.");
  }
  const calls = response.tool_calls ?? [];
  if (calls.length === 0) break;
  if (trace.length + calls.length > maxToolCalls) {
    throw new Error("Tool-call limit reached.");
  }

  for (const call of calls) {
    if (call.name !== executeGremlin.name || !call.id) {
      throw new Error("Unknown tool or missing call ID.");
    }
    console.log("Executing:", call.args.query);
    // Passing arguments invokes the tool's Zod validation and SDR callback.
    const args = await executeGremlin.schema.parseAsync(call.args);
    const result = await executeGremlin.invoke(args);
    if (typeof result !== "string") {
      throw new Error("Expected a JSON tool result.");
    }
    trace.push({ query: String(call.args.query), result: JSON.parse(result) });
    messages.push(
      new ToolMessage({
        content: result,
        tool_call_id: call.id,
        name: call.name,
      }),
    );
  }
  response = await modelWithTools.invoke(messages, {
    signal: AbortSignal.timeout(600_000),
  });
  messages.push(response);
}

if (trace.length === 0) {
  console.log("No tool calls were made; SDR was not queried.");
}
ui.markdown(
  typeof response.content === "string"
    ? response.content
    : JSON.stringify(response.content),
);

Executing: g.V().hasLabel("Study").count()
Executing: g.V().hasLabel("Study").limit(5).values("name")


SDR contains **3088 studies**. Sample of study aliases: **CRD-09-2036, NWS-UW-IGSB, SHA-MC-BXCC, E2E-TA-PRAD, JRR-10-1001**.

In [ ]:
ui.table(trace);


## Inspect and experiment

Change the question to "Show version counts for up to five studies" and rerun
from step 2. Inspect `messages` or `response.usage_metadata` in another cell.

- If the model is missing, install `granite4:7b` on the configured server.
- If Ollama reports unsupported tools, verify the capabilities of that installed
  model.
- If credentials expire, rerun the AWS SSO login command.
- If the hostname is unreachable, check connectivity from the Deno kernel's
  environment.

References: [Deno walkthrough](https://deno.com/blog/the-dino-llama-and-whale),
[LangChain ChatOllama tool calling](https://docs.langchain.com/oss/javascript/integrations/chat/ollama).